# 07 Model Persistence and Inference


In [ ]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

PROJECT_ROOT

## Train, Select, and Save


In [ ]:
from run_model_persistence_and_inference import run_pipeline

outputs = run_pipeline(project_root=str(PROJECT_ROOT))
metadata_df = outputs["metadata_df"]
results_df = outputs["results_df"]

metadata_df

In [ ]:
results_df

## Inspect the Persisted Artifact


In [ ]:
schema_path = PROJECT_ROOT / "models" / "sklearn" / "best_direction_model_schema.json"
model_path = PROJECT_ROOT / "models" / "sklearn" / "best_direction_model.joblib"

schema = json.loads(schema_path.read_text(encoding="utf-8"))
model = joblib.load(model_path)

schema

## Example Future Inference


In [ ]:
dataset_path = PROJECT_ROOT / "data" / "processed" / schema["dataset_name"]
df = pd.read_csv(dataset_path, parse_dates=["trading_date"])

feature_cols = schema["numeric_features"] + schema["categorical_features"]
latest_rows = df.sort_values("trading_date").tail(10).copy()
latest_rows["predicted_probability_up"] = model.predict_proba(latest_rows[feature_cols])[:, 1]
latest_rows["predicted_up"] = model.predict(latest_rows[feature_cols])

latest_rows[["ticker", "trading_date", "target_next_day_up", "predicted_up", "predicted_probability_up"]]